# 04. Final Results

Final comparison and performance benchmark.

In [ ]:
import sys
import torch
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('..')
from rs.model import PositionPredictor, FocalLoss, count_parameters
from rs.training import train_model
from rs.evaluation import  full_evaluation, benchmark_time
from rs.channels import qsc_erasure_channel
from rs.dataset_gen import RSPositionDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parents[0]
table_out_dir = ROOT / "tables"
table_out_dir.mkdir(parents=True, exist_ok=True)

graph_out_dir = ROOT / "graphs"
graph_out_dir.mkdir(parents=True, exist_ok=True)

model_out_dir = ROOT / "models"
model_out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
P_ERR, P_ERASE = 0.02, 0.06
TRAIN_SIZE, EPOCHS = 50000, 500
FOCAL_ALPHA, FOCAL_GAMMA = 0.2, 1.5
THRESHOLD = 0.3

In [ ]:
print('Generating dataset...')
dataset = RSPositionDataset(TRAIN_SIZE, P_ERR, P_ERASE)
print(f'Generated {len(dataset)} examples.')

print('Model training...')
model = PositionPredictor(dropout=0.1).to(device)
criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
losses = train_model(model, dataset, criterion, epochs=EPOCHS, device=device)
print(f'Parameters count: {count_parameters(model):,}')

In [ ]:
results = full_evaluation(model, qsc_erasure_channel, P_ERR, threshold=THRESHOLD, device=device)
df = pd.DataFrame(results)
df.to_csv(table_out_dir / 'final_results.csv', index=False)
print(df.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df['p_erase'], df['classic_hint'], 'g-o', label='Classic+hint')
plt.plot(df['p_erase'], df['hybrid'], 'b-^', label='Hybrid')
plt.plot(df['p_erase'], df['classic'], 'r-s', label='Classic')
plt.fill_between(df['p_erase'], df['classic'], df['hybrid'], alpha=0.3)
plt.xlabel('p_erase'); plt.ylabel('FSR'); plt.legend(); plt.grid(True, alpha=0.3)
plt.title('Сравнение декодеров RS(255, 223)')

plt.savefig(graph_out_dir / 'final_results.png', dpi=150); plt.show()

In [ ]:
row = df[df['p_erase'] == 0.06].iloc[0]
gap_filled = (row['hybrid'] - row['classic']) / (row['classic_hint'] - row['classic'])
improvement = row['hybrid'] / row['classic']

print(f"При p_erase = 0.06:")
print(f"  Classic:      {row['classic']:.1%}")
print(f"  Hybrid:       {row['hybrid']:.1%}")
print(f"  Classic+hint: {row['classic_hint']:.1%}")
print(f"  Gap filled:   {gap_filled:.0%}")
print(f"  Improvement:  {improvement:.1f}x")

In [ ]:
print('Performance benchmark...')
timing = benchmark_time(model, qsc_erasure_channel, P_ERR, P_ERASE, 
                        threshold=THRESHOLD, num_cycles=10, msgs_per_cycle=100, device=device)

print(f"Classic: {timing['classic_mean']*1000:.1f} ± {timing['classic_std']*1000:.1f} ms / 100 msgs")
print(f"Hybrid:  {timing['hybrid_mean']*1000:.1f} ± {timing['hybrid_std']*1000:.1f} ms / 100 msgs")
print(f"Classic + hint: {timing['classic_w_hint_mean']*1000:.1f} ± {timing['classic_w_hint_std']*1000:.1f} ms / 100 msgs")
print(f"Slowdown relative to classic: {timing['slowdown']:.1f}x")
print(f"Slowdown relative to classic + hint: {timing['slowdown_w_hints']}")

df = pd.DataFrame(timing, index=[0])
df.to_csv(table_out_dir / 'benchmarking.csv', index=False)

In [ ]:
torch.save(model.state_dict(), model_out_dir / 'model_final.pth')